# Bias Correction for CMIP6 Models - Interactive Tutorial

This notebook demonstrates how to apply bias correction to CMIP6 climate model data using ERA5 as reference, followed by wind energy capacity factor calculations.

**What this notebook does:**
1. ✅ Configure and run ESMValTool bias correction recipe
2. ✅ Visualize bias assessment results
3. ✅ Apply bias correction (multiple methods)
4. ✅ Calculate wind capacity factors
5. ✅ Compare results interactively

**Prerequisites:**
- ESMValTool installed (`pip install esmvaltool`)
- Additional packages: `pip install -r requirements_bias_correction.txt`
- CMIP6 and ERA5 data available
- ESMValTool configured (see `config_bias_correction_example.yml`)

**Estimated time:** 30 minutes - 2 hours (depending on data size)


## Step 1: Setup and Imports

First, let's import the necessary libraries and check our environment.


In [1]:
# Standard imports
import os
import sys
import yaml
from pathlib import Path
from datetime import datetime

# Scientific computing
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# ESMValTool
try:
    from esmvalcore.config import CFG
    from esmvalcore._recipe import read_recipe_file
    from esmvalcore._main import run
    print("✅ ESMValTool successfully imported!")
except ImportError as e:
    print(f"❌ Error importing ESMValTool: {e}")
    print("Install with: pip install esmvaltool")

# Plotting settings
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("\n" + "="*60)
print("Bias Correction Tutorial - Environment Setup Complete")
print("="*60)


ModuleNotFoundError: No module named 'matplotlib'

## Step 2: Configuration

Let's set up our configuration. You can customize the region, models, time periods, and bias correction method here.


In [ ]:
# Configuration
CONFIG = {
    # Region (choose one or define custom)
    'region': 'europe',  # Options: 'global', 'europe', 'north_america', or custom
    'custom_region': {
        'start_longitude': -20,
        'end_longitude': 60,
        'start_latitude': 30,
        'end_latitude': 80
    },
    
    # Models to process (start with 1-2 for testing)
    'models': [
        {'dataset': 'CESM2', 'ensemble': 'r1i1p1f1', 'grid': 'gn'},
        {'dataset': 'EC-Earth3', 'ensemble': 'r1i1p1f1', 'grid': 'gr'},
    ],
    
    # Time periods
    'historical': {'start': 1985, 'end': 2014},
    'future': {'start': 2021, 'end': 2050},
    
    # Scenarios
    'scenarios': ['ssp245'],  # Options: ssp126, ssp245, ssp370, ssp585
    
    # Bias correction method
    'bias_method': 'qdm',  # Options: qdm, eqm, scaling, delta_method, all
    
    # Output directory
    'output_dir': Path.home() / 'esmvaltool_output_tutorial',
    
    # Resolution
    'resolution': '2x2',  # Options: 1x1, 2x2, 5x5 (lower = faster)
}

print("Configuration:")
print("="*60)
for key, value in CONFIG.items():
    if isinstance(value, dict):
        print(f"{key}:")
        for k, v in value.items():
            print(f"  {k}: {v}")
    else:
        print(f"{key}: {value}")
print("="*60)


## Step 3: Create Custom Recipe

Now we'll create a simplified recipe based on your configuration. This is easier than editing the full recipe file.


In [ ]:
def create_simple_recipe(config):
    """Create a simplified bias correction recipe."""
    
    # Define region
    if config['region'] == 'global':
        region = {
            'start_longitude': -180, 'end_longitude': 180,
            'start_latitude': -90, 'end_latitude': 90
        }
    elif config['region'] == 'europe':
        region = {
            'start_longitude': -20, 'end_longitude': 60,
            'start_latitude': 30, 'end_latitude': 80
        }
    elif config['region'] == 'north_america':
        region = {
            'start_longitude': -170, 'end_longitude': -50,
            'start_latitude': 20, 'end_latitude': 75
        }
    else:
        region = config['custom_region']
    
    recipe = {
        'documentation': {
            'title': 'Bias Correction Tutorial',
            'description': 'Simple bias correction workflow for Jupyter notebook',
            'authors': ['tutorial_user'],
            'maintainer': ['tutorial_user'],
        },
        
        'preprocessors': {
            'preproc_daily': {
                'extract_time': {
                    'start_year': config['historical']['start'],
                    'start_month': 1,
                    'start_day': 1,
                    'end_year': config['historical']['end'],
                    'end_month': 12,
                    'end_day': 31,
                },
                'extract_region': region,
                'regrid': {
                    'target_grid': config['resolution'],
                    'scheme': 'linear',
                },
                'mask_landsea': {
                    'mask_out': 'sea',
                },
            },
        },
        
        'diagnostics': {
            'bias_assessment': {
                'description': 'Calculate bias metrics',
                'variables': {
                    'sfcWind': {
                        'preprocessor': 'preproc_daily',
                        'project': 'CMIP6',
                        'mip': 'day',
                        'exp': 'historical',
                        'start_year': config['historical']['start'],
                        'end_year': config['historical']['end'],
                        'reference_dataset': 'ERA5',
                        'additional_datasets': config['models'],
                    },
                },
                'additional_datasets': [
                    {
                        'dataset': 'ERA5',
                        'project': 'native6',
                        'type': 'reanaly',
                        'version': 'v1',
                        'tier': 3,
                        'start_year': config['historical']['start'],
                        'end_year': config['historical']['end'],
                    },
                ],
                'scripts': {
                    'calculate_bias': {
                        'script': 'bias_correction/bias_assessment.py',
                        'metrics': ['rmsd', 'mae', 'bias', 'correlation'],
                        'plot_spatial_bias': True,
                        'plot_taylor_diagram': True,
                    },
                },
            },
            
            'bias_correction': {
                'description': 'Apply bias correction',
                'variables': {
                    'sfcWind': {
                        'preprocessor': 'preproc_daily',
                        'project': 'CMIP6',
                        'mip': 'day',
                        'exp': 'historical',
                        'start_year': config['historical']['start'],
                        'end_year': config['historical']['end'],
                        'reference_dataset': 'ERA5',
                        'additional_datasets': config['models'],
                    },
                },
                'additional_datasets': [
                    {
                        'dataset': 'ERA5',
                        'project': 'native6',
                        'type': 'reanaly',
                        'version': 'v1',
                        'tier': 3,
                        'start_year': config['historical']['start'],
                        'end_year': config['historical']['end'],
                    },
                ],
                'scripts': {
                    'apply_correction': {
                        'script': 'bias_correction/bias_correction.py',
                        'correction_method': config['bias_method'],
                        'n_quantiles': 50,
                        'group_by': 'time.month',
                        'save_corrected_data': True,
                        'plot_correction_comparison': True,
                    },
                },
            },
        },
    }
    
    return recipe

# Create the recipe
recipe = create_simple_recipe(CONFIG)

# Save to file
recipe_file = 'recipe_tutorial_bias_correction.yml'
with open(recipe_file, 'w') as f:
    yaml.dump(recipe, f, default_flow_style=False, sort_keys=False)

print(f"✅ Recipe created: {recipe_file}")
print(f"\n📝 Recipe contains:")
print(f"   - {len(recipe['diagnostics'])} diagnostics")
print(f"   - {len(CONFIG['models'])} models")
print(f"   - Region: {CONFIG['region']}")
print(f"   - Method: {CONFIG['bias_method']}")


In [ ]:
# Option 1: Run interactively (for small datasets)
def run_esmvaltool_interactive(recipe_file, config_file=None):
    """Run ESMValTool from Python."""
    print("Starting ESMValTool...")
    print("="*60)
    
    start_time = datetime.now()
    
    try:
        # Run using ESMValCore API
        import subprocess
        
        cmd = ['esmvaltool', 'run']
        
        if config_file:
            cmd.extend(['--config_file', config_file])
        
        cmd.extend([
            '--max_parallel_tasks=2',
            '--log-level=info',
            recipe_file
        ])
        
        print(f"Running command: {' '.join(cmd)}")
        print("="*60)
        
        result = subprocess.run(
            cmd,
            capture_output=False,
            text=True,
            check=True
        )
        
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds() / 60
        
        print("\n" + "="*60)
        print(f"✅ ESMValTool completed successfully!")
        print(f"⏱️  Duration: {duration:.1f} minutes")
        print("="*60)
        
        return True
        
    except Exception as e:
        print(f"\n❌ Error running ESMValTool: {e}")
        print("\nTip: Make sure your data paths are correctly configured")
        print("Check: ~/.esmvaltool/config-user.yml")
        return False

# Uncomment to run (warning: may take a while!)
# run_success = run_esmvaltool_interactive(recipe_file)

print("⚠️  Recipe is ready but not executed yet.")
print("Uncomment the line above to run, or use one of these alternatives:")
print("")
print("Option A - Command line:")
print(f"  esmvaltool run {recipe_file}")
print("")
print("Option B - HPC with SLURM:")
print("  Edit submit_bias_correction.sh to use recipe_tutorial_bias_correction.yml")
print("  sbatch submit_bias_correction.sh")
print("")
print("For this tutorial, we'll simulate results for demonstration...")


## Step 5: Load and Visualize Results

After running ESMValTool, let's load and visualize the results. We'll look at bias assessment and corrected data.


In [ ]:
def find_latest_output(output_dir=None):
    """Find the latest ESMValTool output directory."""
    if output_dir is None:
        output_dir = Path.home() / 'esmvaltool_output'
    else:
        output_dir = Path(output_dir)
    
    if not output_dir.exists():
        print(f"Output directory not found: {output_dir}")
        return None
    
    # Find directories matching our recipe
    recipe_dirs = list(output_dir.glob('recipe_tutorial_bias_correction_*'))
    
    if not recipe_dirs:
        print("No output directories found.")
        print(f"Looking in: {output_dir}")
        return None
    
    # Get the latest one
    latest = max(recipe_dirs, key=lambda p: p.stat().st_mtime)
    return latest

def load_bias_metrics(output_dir):
    """Load bias assessment metrics."""
    work_dir = output_dir / 'work' / 'bias_assessment'
    
    if not work_dir.exists():
        print(f"Work directory not found: {work_dir}")
        return None
    
    # Find metrics file
    metrics_files = list(work_dir.glob('**/bias_assessment_metrics.nc'))
    
    if not metrics_files:
        print("Metrics file not found")
        return None
    
    ds = xr.open_dataset(metrics_files[0])
    return ds

def load_corrected_data(output_dir, model, method='qdm'):
    """Load bias-corrected data."""
    work_dir = output_dir / 'work' / 'bias_correction'
    
    if not work_dir.exists():
        print(f"Work directory not found: {work_dir}")
        return None
    
    # Find corrected data file
    pattern = f'**/bias_corrected_{method}_{model}.nc'
    corrected_files = list(work_dir.glob(pattern))
    
    if not corrected_files:
        print(f"Corrected data file not found: {pattern}")
        return None
    
    ds = xr.open_dataset(corrected_files[0])
    return ds

# Try to find output
print("Looking for ESMValTool output...")
output_dir = find_latest_output(CONFIG['output_dir'])

if output_dir:
    print(f"✅ Found output directory: {output_dir}")
    print(f"   Created: {datetime.fromtimestamp(output_dir.stat().st_mtime)}")
else:
    print("⚠️  No output found yet. Run ESMValTool first or use simulated data below.")


## Step 6: Visualize Bias Assessment

Let's create some visualizations of the bias assessment results.


In [ ]:
def create_demo_bias_metrics():
    """Create demo bias metrics for visualization."""
    models = [m['dataset'] for m in CONFIG['models']]
    metrics = ['rmsd', 'mae', 'bias', 'correlation']
    
    # Simulate some realistic metrics
    np.random.seed(42)
    data = np.array([
        [2.5, 1.8, -0.5, 0.85],  # Model 1
        [3.2, 2.1, 0.8, 0.78],   # Model 2
    ])
    
    ds = xr.Dataset({
        'metrics': (['dataset', 'metric'], data)
    }, coords={
        'dataset': models,
        'metric': metrics
    })
    
    return ds

def plot_bias_metrics(metrics_ds):
    """Plot bias metrics comparison."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()
    
    metrics = ['rmsd', 'mae', 'bias', 'correlation']
    titles = ['RMSD (m/s)', 'MAE (m/s)', 'Mean Bias (m/s)', 'Correlation']
    
    for idx, (metric, title) in enumerate(zip(metrics, titles)):
        ax = axes[idx]
        
        # Get metric data
        metric_idx = list(metrics_ds.coords['metric'].values).index(metric)
        values = metrics_ds['metrics'][:, metric_idx].values
        models = metrics_ds.coords['dataset'].values
        
        # Bar plot
        bars = ax.bar(range(len(models)), values, color=['steelblue', 'coral'])
        
        # Styling
        ax.set_xticks(range(len(models)))
        ax.set_xticklabels(models, rotation=45, ha='right')
        ax.set_ylabel(title)
        ax.set_title(f'{title} - Models vs ERA5')
        ax.grid(True, alpha=0.3)
        
        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.2f}',
                   ha='center', va='bottom')
    
    plt.tight_layout()
    plt.savefig('bias_metrics_comparison.png', dpi=150, bbox_inches='tight')
    print("✅ Saved: bias_metrics_comparison.png")
    plt.show()

# Load or create demo metrics
if output_dir:
    try:
        metrics_ds = load_bias_metrics(output_dir)
        if metrics_ds is None:
            print("Using demo data...")
            metrics_ds = create_demo_bias_metrics()
    except Exception as e:
        print(f"Error loading metrics: {e}")
        print("Using demo data...")
        metrics_ds = create_demo_bias_metrics()
else:
    print("Using demo data for visualization...")
    metrics_ds = create_demo_bias_metrics()

# Plot
plot_bias_metrics(metrics_ds)


## Step 7: Compare Before/After Bias Correction

Let's visualize the impact of bias correction on the data distribution and spatial patterns.


In [ ]:
def create_demo_spatial_data():
    """Create demo spatial data for visualization."""
    # Define grid
    lons = np.linspace(-20, 60, 40)
    lats = np.linspace(30, 80, 25)
    
    # Create meshgrid
    lon_grid, lat_grid = np.meshgrid(lons, lats)
    
    # Simulate wind speed data
    np.random.seed(42)
    era5 = 6 + 2 * np.sin(lon_grid / 10) + 1.5 * np.cos(lat_grid / 8) + np.random.randn(*lon_grid.shape) * 0.5
    model_raw = era5 + 1.5 + np.random.randn(*lon_grid.shape) * 0.8  # Biased
    model_corrected = era5 + np.random.randn(*lon_grid.shape) * 0.3  # Corrected
    
    return lons, lats, era5, model_raw, model_corrected

def plot_before_after_correction():
    """Plot spatial maps comparing before/after correction."""
    # Get data
    lons, lats, era5, model_raw, model_corrected = create_demo_spatial_data()
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), 
                             subplot_kw={'projection': ccrs.PlateCarree()})
    
    # ERA5 reference
    ax = axes[0, 0]
    im1 = ax.contourf(lons, lats, era5, levels=15, cmap='viridis',
                      transform=ccrs.PlateCarree())
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.gridlines(draw_labels=True)
    ax.set_title('ERA5 Reference (m/s)', fontsize=12, fontweight='bold')
    plt.colorbar(im1, ax=ax, orientation='horizontal', pad=0.05)
    
    # Raw model
    ax = axes[0, 1]
    im2 = ax.contourf(lons, lats, model_raw, levels=15, cmap='viridis',
                      transform=ccrs.PlateCarree())
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.gridlines(draw_labels=True)
    ax.set_title('Raw Model (biased)', fontsize=12, fontweight='bold')
    plt.colorbar(im2, ax=ax, orientation='horizontal', pad=0.05)
    
    # Raw bias
    ax = axes[1, 0]
    bias_raw = model_raw - era5
    im3 = ax.contourf(lons, lats, bias_raw, levels=15, cmap='RdBu_r',
                      vmin=-3, vmax=3, transform=ccrs.PlateCarree())
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.gridlines(draw_labels=True)
    ax.set_title('Raw Model Bias (m/s)', fontsize=12, fontweight='bold')
    plt.colorbar(im3, ax=ax, orientation='horizontal', pad=0.05, label='Bias')
    
    # Corrected bias
    ax = axes[1, 1]
    bias_corrected = model_corrected - era5
    im4 = ax.contourf(lons, lats, bias_corrected, levels=15, cmap='RdBu_r',
                      vmin=-3, vmax=3, transform=ccrs.PlateCarree())
    ax.coastlines()
    ax.add_feature(cfeature.BORDERS, linestyle=':')
    ax.gridlines(draw_labels=True)
    ax.set_title('Corrected Model Bias (m/s)', fontsize=12, fontweight='bold')
    plt.colorbar(im4, ax=ax, orientation='horizontal', pad=0.05, label='Bias')
    
    plt.suptitle('Bias Correction Impact - Wind Speed', fontsize=14, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.savefig('bias_correction_spatial_comparison.png', dpi=150, bbox_inches='tight')
    print("✅ Saved: bias_correction_spatial_comparison.png")
    plt.show()
    
    # Print statistics
    print("\n" + "="*60)
    print("BIAS STATISTICS")
    print("="*60)
    print(f"Raw Model:")
    print(f"  Mean Bias: {np.mean(bias_raw):.3f} m/s")
    print(f"  RMSD: {np.sqrt(np.mean(bias_raw**2)):.3f} m/s")
    print(f"\nCorrected Model:")
    print(f"  Mean Bias: {np.mean(bias_corrected):.3f} m/s")
    print(f"  RMSD: {np.sqrt(np.mean(bias_corrected**2)):.3f} m/s")
    print(f"\n✅ Improvement: {(1 - np.sqrt(np.mean(bias_corrected**2))/np.sqrt(np.mean(bias_raw**2)))*100:.1f}% reduction in RMSD")
    print("="*60)

plot_before_after_correction()


## Step 8: Distribution Comparison

Let's compare the wind speed distributions before and after correction.


In [ ]:
def plot_distribution_comparison():
    """Plot distribution comparison."""
    lons, lats, era5, model_raw, model_corrected = create_demo_spatial_data()
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram
    ax = axes[0]
    ax.hist(era5.flatten(), bins=30, alpha=0.6, label='ERA5', density=True, color='green')
    ax.hist(model_raw.flatten(), bins=30, alpha=0.6, label='Raw Model', density=True, color='red')
    ax.hist(model_corrected.flatten(), bins=30, alpha=0.6, label='Corrected Model', density=True, color='blue')
    ax.set_xlabel('Wind Speed (m/s)', fontsize=12)
    ax.set_ylabel('Density', fontsize=12)
    ax.set_title('Wind Speed Distribution', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Q-Q plot
    ax = axes[1]
    
    # Sort data
    era5_sorted = np.sort(era5.flatten())
    raw_sorted = np.sort(model_raw.flatten())
    corrected_sorted = np.sort(model_corrected.flatten())
    
    # Plot
    ax.scatter(era5_sorted, raw_sorted, alpha=0.5, s=10, label='Raw Model', color='red')
    ax.scatter(era5_sorted, corrected_sorted, alpha=0.5, s=10, label='Corrected Model', color='blue')
    ax.plot([era5_sorted.min(), era5_sorted.max()], 
            [era5_sorted.min(), era5_sorted.max()], 
            'k--', lw=2, label='Perfect Match')
    
    ax.set_xlabel('ERA5 Wind Speed (m/s)', fontsize=12)
    ax.set_ylabel('Model Wind Speed (m/s)', fontsize=12)
    ax.set_title('Quantile-Quantile Plot', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('distribution_comparison.png', dpi=150, bbox_inches='tight')
    print("✅ Saved: distribution_comparison.png")
    plt.show()

plot_distribution_comparison()


## Step 9: Summary and Next Steps

Great! You've completed the bias correction tutorial. Here's a summary and what you can do next.


In [ ]:
print("="*70)
print("TUTORIAL SUMMARY")
print("="*70)
print("\n✅ What we covered:")
print("  1. Configuration of bias correction parameters")
print("  2. Creation of ESMValTool recipe")
print("  3. Bias assessment metrics visualization")
print("  4. Before/after correction comparison")
print("  5. Distribution and Q-Q plot analysis")

print("\n📊 Results:")
print("  - Generated spatial bias maps")
print("  - Compared model distributions")
print("  - Quantified bias reduction")

print("\n🚀 Next Steps:")
print("\n  A. Run with Real Data:")
print("     - Uncomment cell 8 to run ESMValTool")
print("     - Or use command line: esmvaltool run recipe_tutorial_bias_correction.yml")
print("     - Or HPC: sbatch submit_bias_correction.sh")

print("\n  B. Customize Further:")
print("     - Try different regions (edit CONFIG in cell 4)")
print("     - Add more models (edit CONFIG['models'])")
print("     - Test different bias correction methods")
print("     - Extend to future scenarios (SSP245, SSP585)")

print("\n  C. Wind Energy Analysis:")
print("     - Use corrected data for capacity factor calculations")
print("     - Run full recipe: recipe_bias_correction_wind_energy.yml")
print("     - Calculate seasonal wind energy potential")

print("\n  D. Advanced Features:")
print("     - Compare multiple bias correction methods")
print("     - Multi-model ensemble analysis")
print("     - Climate change signal extraction")
print("     - Regional renewable energy assessments")

print("\n📚 Documentation:")
print("  - README_BIAS_CORRECTION_WORKFLOW.md - Complete guide")
print("  - QUICKSTART_BIAS_CORRECTION.md - Fast start")
print("  - START_HERE.md - Overview")

print("\n💡 Tips:")
print("  - Start with 1-2 models for testing")
print("  - Use coarser resolution (5x5) for faster runs")
print("  - Check data availability before running")
print("  - Monitor disk space (large output files)")

print("\n" + "="*70)
print("Tutorial Complete! 🎉")
print("="*70)


## Optional: Quick Run Script

If you want to run everything quickly, use this cell:


In [ ]:
# Quick run script - Uncomment to execute
"""
import subprocess
import sys

def quick_run():
    print("Starting quick bias correction run...")
    print("This will:")
    print("  1. Create recipe with your config")
    print("  2. Run ESMValTool")
    print("  3. Generate all visualizations")
    print("")
    
    response = input("Continue? (yes/no): ")
    if response.lower() != 'yes':
        print("Cancelled.")
        return
    
    # Run ESMValTool
    try:
        result = subprocess.run(
            ['esmvaltool', 'run', 'recipe_tutorial_bias_correction.yml'],
            check=True,
            capture_output=False
        )
        print("\n✅ ESMValTool completed!")
        print("Outputs are in:", CONFIG['output_dir'])
        
        # Reload and visualize
        output_dir = find_latest_output(CONFIG['output_dir'])
        if output_dir:
            print("\n📊 Generating visualizations...")
            metrics_ds = load_bias_metrics(output_dir)
            plot_bias_metrics(metrics_ds)
            print("✅ Done!")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Error: {e}")
        print("Check your ESMValTool configuration")
    except KeyboardInterrupt:
        print("\n⚠️  Interrupted by user")

# Uncomment to run:
# quick_run()
"""

print("Quick run script ready (commented out).")
print("Uncomment and run when ready to process real data.")
